# Landsat yearly QA-best preview gallery

This notebook scans every LC08 T1 scene under `s3://usgs-landsat/collection02/level-2/standard/oli-tirs/<year>/<path>/<row>/` between 1984 and 2025, downloads their `thumb_large` previews and `QA_PIXEL` rasters into `../data/landsat/<year>-<path-row>/`, scores each scene using the QA bits, and finally plots the best preview per year.

In [1]:
import boto3
import numpy as np
import rasterio
from pathlib import Path
import matplotlib.pyplot as plt
from datetime import datetime
import math

In [13]:
start_year = 2013
end_year = 2025
target_path = "216"
target_row = "074"
product_category = "T1"
spacecraft_prefix = "LC08"
aws_registry_url = "https://registry.opendata.aws/usgs-landsat/index.html"
bucket_name = "usgs-landsat"
collection_template = "collection02/level-2/standard/oli-tirs/{year}/{path}/{row}/"
requester_pays_args = {"RequestPayer": "requester"}

local_data_root = Path("../data/landsat")
local_data_root.mkdir(parents=True, exist_ok=True)

s3_client = boto3.client("s3")
print(f"Prepared local storage at {local_data_root.resolve()}")
print(f"Registry reference: {aws_registry_url}")
print(f"Years: {start_year}-{end_year}, path/row {target_path}/{target_row}, scenes from {spacecraft_prefix}")

Prepared local storage at /Volumes/workplace/Projects/carbono-data-collection/data/landsat
Registry reference: https://registry.opendata.aws/usgs-landsat/index.html
Years: 2013-2025, path/row 216/074, scenes from LC08


In [12]:
yearly_assets = {}
paginator = s3_client.get_paginator("list_objects_v2")

for year in range(start_year, end_year + 1):
    collection_prefix = collection_template.format(year=year, path=target_path, row=target_row)
    local_dir = local_data_root / f"{year}-{target_path}-{target_row}"
    local_dir.mkdir(parents=True, exist_ok=True)
    thumb_candidates = []
    seen_scene_ids = set()
    print(bucket_name + "/" + collection_prefix)
    print(requester_pays_args)

    for page in paginator.paginate(Bucket=bucket_name, Prefix=collection_prefix, **requester_pays_args):
        for obj in page.get("Contents", []):
            #print(obj)
            key = obj["Key"]
            if not key.endswith("_thumb_large.jpg") and not key.endswith("_thumb_large.jpeg"):
                continue
            if f"_{product_category}/" not in key:
                continue
            scene_id = key.split("/")[-2]
            if not scene_id.startswith(spacecraft_prefix):
                continue
            if scene_id in seen_scene_ids:
                continue
            seen_scene_ids.add(scene_id)
            tokens = scene_id.split("_")
            capture_date = None
            if len(tokens) > 3:
                try:
                    capture_date = datetime.strptime(tokens[3], "%Y%m%d")
                except ValueError:
                    capture_date = None
            #print(obj)
            thumb_candidates.append({
                "year": year,
                "key": key,
                "scene_id": scene_id,
                "capture_date": capture_date,
                "collection_prefix": collection_prefix,
                "local_dir": local_dir,
            })

    thumb_candidates.sort(key=lambda item: item["capture_date"] or datetime.min)
    yearly_assets[year] = thumb_candidates
    print(f"Year {year}: found {len(thumb_candidates)} LC08 {product_category} thumbnails")

usgs-landsat/collection02/level-2/standard/oli-tirs/1984/216/074/
{'RequestPayer': 'requester'}
Year 1984: found 0 LC08 T1 thumbnails
usgs-landsat/collection02/level-2/standard/oli-tirs/1985/216/074/
{'RequestPayer': 'requester'}
Year 1985: found 0 LC08 T1 thumbnails
usgs-landsat/collection02/level-2/standard/oli-tirs/1986/216/074/
{'RequestPayer': 'requester'}
Year 1986: found 0 LC08 T1 thumbnails
usgs-landsat/collection02/level-2/standard/oli-tirs/1987/216/074/
{'RequestPayer': 'requester'}
Year 1987: found 0 LC08 T1 thumbnails
usgs-landsat/collection02/level-2/standard/oli-tirs/1988/216/074/
{'RequestPayer': 'requester'}
Year 1988: found 0 LC08 T1 thumbnails
usgs-landsat/collection02/level-2/standard/oli-tirs/1989/216/074/
{'RequestPayer': 'requester'}
Year 1989: found 0 LC08 T1 thumbnails
usgs-landsat/collection02/level-2/standard/oli-tirs/1990/216/074/
{'RequestPayer': 'requester'}
Year 1990: found 0 LC08 T1 thumbnails
usgs-landsat/collection02/level-2/standard/oli-tirs/1991/216/0

In [2]:
best_per_year = []

for year in range(start_year, end_year + 1):
    assets = yearly_assets.get(year, [])
    if not assets:
        print(f"Year {year}: no candidates found")
        continue

    year_results = []
    print(f"Processing {len(assets)} scenes for {year}")

    for asset in assets:
        scene_id = asset["scene_id"]
        scene_dir = asset["local_dir"]
        thumb_name = asset["key"].split("/")[-1]
        local_thumb = scene_dir / thumb_name
        asset["thumb_path"] = local_thumb

        if local_thumb.exists():
            print(f" - thumb cached for {scene_id}")
        else:
            s3_client.download_file(
                Bucket=bucket_name,
                Key=asset["key"],
                Filename=str(local_thumb),
                ExtraArgs=requester_pays_args,
            )
            print(f" - downloaded thumb {thumb_name}")

        qa_key = f"{asset['collection_prefix']}{scene_id}/{scene_id}_QA_PIXEL.TIF"
        local_qa = scene_dir / f"{scene_id}_QA_PIXEL.TIF"
        asset["qa_path"] = local_qa

        if not local_qa.exists():
            try:
                s3_client.download_file(
                    Bucket=bucket_name,
                    Key=qa_key,
                    Filename=str(local_qa),
                    ExtraArgs=requester_pays_args,
                )
                print(f"   downloaded QA_PIXEL for {scene_id}")
            except s3_client.exceptions.NoSuchKey:
                print(f"   QA_PIXEL missing at {qa_key}")
                continue
            except Exception as exc:
                print(f"   failed to download QA for {scene_id}: {exc}")
                continue
        else:
            print(f"   QA_PIXEL cached for {scene_id}")

        try:
            with rasterio.open(local_qa) as src:
                qa_array = src.read(1)
        except Exception as exc:
            print(f"   failed to read QA for {scene_id}: {exc}")
            continue

        fill_mask = (qa_array & 1) != 0
        dilated_mask = (qa_array & (1 << 1)) != 0
        cirrus_mask = (qa_array & (1 << 2)) != 0
        cloud_mask = (qa_array & (1 << 3)) != 0
        shadow_mask = (qa_array & (1 << 4)) != 0

        valid_pixels = qa_array.size - int(fill_mask.sum())
        if valid_pixels <= 0:
            print(f"   no valid pixels for {scene_id}")
            continue

        problem_mask = dilated_mask | cirrus_mask | cloud_mask | shadow_mask
        cloud_pixels = int((problem_mask & (~fill_mask)).sum())
        quality_score = 1 - (cloud_pixels / valid_pixels)

        year_results.append({
            "year": year,
            "scene_id": scene_id,
            "capture_date": asset.get("capture_date"),
            "thumb_path": local_thumb,
            "quality_score": quality_score,
        })
        print(f"   QA score {quality_score:.4f} for {scene_id}")

    if not year_results:
        print(f"No QA results for {year}")
        continue

    year_results.sort(key=lambda item: item["quality_score"], reverse=True)
    best = year_results[0]
    best_per_year.append(best)
    best_date = best["capture_date"].strftime("%Y-%m-%d") if best["capture_date"] else "unknown"
    print(f"Best {year}: {best['scene_id']} (captured {best_date}) -> {best['quality_score']:.4f}")

NameError: name 'start_year' is not defined

In [1]:
if not best_per_year:
    raise RuntimeError("No yearly best scenes were determined.")

best_per_year.sort(key=lambda item: item["year"])
print(f"Computed best previews for {len(best_per_year)} years")
for record in best_per_year:
    date_label = record["capture_date"].strftime("%Y-%m-%d") if record["capture_date"] else "unknown"
    print(f" - {record['year']}: {record['scene_id']} ({date_label}) score {record['quality_score']:.4f}")

n_assets = len(best_per_year)
n_cols = min(3, max(1, n_assets))
n_rows = math.ceil(n_assets / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.5 * n_cols, 4.5 * n_rows))
axes = np.array(axes).reshape(-1)

for ax in axes[n_assets:]:
    ax.axis('off')

for ax, asset in zip(axes, best_per_year):
    image = plt.imread(asset["thumb_path"])
    date_label = asset["capture_date"].strftime("%Y-%m-%d") if asset["capture_date"] else "unknown"
    ax.imshow(image)
    ax.set_title(f"{asset['year']} - qa={asset['quality_score']:.3f}")
    ax.axis('off')

plt.tight_layout()
panel_output = "../output" / f"best-{spacecraft_prefix.lower()}-{target_path}-{target_row}-{start_year}-{end_year}.png"
fig.savefig(panel_output, dpi=300)
print(f"Saved panel figure to {panel_output}")

NameError: name 'best_per_year' is not defined